In [1]:
import joblib
from scipy import sparse
import numpy as np


In [10]:
xgb_model = joblib.load("xgboost_tuned.pkl")

# --- Load scraped data ---
X_scraped_tfidf = joblib.load("../feature_engineering/scraped_tfidf_matrix.pkl")  # sparse matrix
X_scraped_struct = joblib.load("../feature_engineering/scraped_feature_list.pkl") # DataFrame 

tfidf = joblib.load("../feature_engineering/tfidf_vectorizer.pkl")
svd = joblib.load("tfidf_svd.pkl")

#need to transform with same svd as tested on
X_scraped_tfidf_svd = svd.transform(X_scraped_tfidf)

In [4]:
def to_numeric_sparse(df):
    df = df.copy()
    bool_cols = df.select_dtypes(include="bool").columns.tolist()
    if bool_cols:
        df[bool_cols] = df[bool_cols].astype(np.int8)
    return sparse.csr_matrix(df.values)

X_scraped_struct_sparse = to_numeric_sparse(X_scraped_struct)

# Convert SVD output to sparse (optional)
X_scraped_tfidf_sparse = sparse.csr_matrix(X_scraped_tfidf_svd)

# Combine SVD + structured features
X_scraped_combined = sparse.hstack([X_scraped_tfidf_sparse, X_scraped_struct_sparse])


In [6]:
y_scraped_pred = xgb_model.predict(X_scraped_combined)
y_scraped_proba = xgb_model.predict_proba(X_scraped_combined)[:, 1]

fraud_indices = np.where(y_scraped_pred == 1)[0]
print(f"Predicted {len(fraud_indices)} fraudulent jobs in scraped data.")


Predicted 1462 fraudulent jobs in scraped data.


In [8]:
num_jobs = X_scraped_struct.shape[0]
num_jobs

8227

In [9]:
print(1462/8227)

0.17770754831651878


Feature Importance Section:

In [13]:
# Structured features
struct_feature_names = X_scraped_struct.columns if hasattr(X_scraped_struct, "columns") else [f"feat_{i}" for i in range(X_scraped_struct.shape[1])]

# Number of SVD components
n_svd = svd.n_components

# ================== 2. Get feature importances ==================
importances = xgb_model.feature_importances_

# Split into SVD vs structured
svd_importances = importances[:n_svd]
struct_importances = importances[n_svd:]

# ------------------ Top structured features ------------------
print("Top Structured Features Influencing Fraud Predictions:\n")
top_struct_idx = np.argsort(struct_importances)[::-1][:15]  # top 15
for i in top_struct_idx:
    print(f"{struct_feature_names[i]}: {struct_importances[i]:.6f}")

# ------------------ Top SVD components ------------------
print("\nTop SVD Components Influencing Fraud Predictions:\n")
top_svd_idx = np.argsort(svd_importances)[::-1][:10]  # top 10 components
for comp_idx in top_svd_idx:
    component = svd.components_[comp_idx]  # shape = number of original TF-IDF features
    top_words_idx = np.argsort(component)[-10:][::-1]  # top 10 words for this component
    top_words = tfidf.get_feature_names_out()[top_words_idx]
    print(f"SVD component {comp_idx} (importance {svd_importances[comp_idx]:.6f}):")
    print("   Top words:", ", ".join(top_words))
    print()


Top Structured Features Influencing Fraud Predictions:

industry_group_Retail/Hospitality: 0.328566
industry_group_Healthcare: 0.011343
has_state_code: 0.005379
industry_group_Other Business/Services: 0.003498
starts_with_direction: 0.003297
industry_group_Government/Nonprofit: 0.002177
has_us_prefix: 0.001895
loc_len: 0.001532
industry_group_Manufacturing/Industrial: 0.001480
industry_group_Engineering/Construction: 0.001440
industry_group_Other: 0.001332
contains_digits: 0.001037
industry_group_Education: 0.000754
location_legitimacy: 0.000324
loc_word_count: 0.000000

Top SVD Components Influencing Fraud Predictions:

SVD component 0 (importance 0.213021):
   Top words: flexible hours, flexible, hours, week immediate, 5000 week, earn 5000, hiring contact, immediate hiring, required flexible, 5000

SVD component 1 (importance 0.047177):
   Top words: experience, team, work, business, sales, customer, company, development, marketing, skills

SVD component 3 (importance 0.014769):
   T

In [14]:
#================== 4. Sample 50 random fraudulent jobs ==================
if len(fraud_indices) > 50:
    sampled_idx = np.random.choice(fraud_indices, size=50, replace=False)
else:
    sampled_idx = fraud_indices

# ================== 5. Show per-job top features ==================
for i, idx in enumerate(sampled_idx):
    print(f"\n--- Fraudulent Job #{i+1} (index {idx}) ---")
    
    # Structured features contribution (rough)
    struct_values = X_scraped_struct.iloc[idx].values
    top_struct_idx = np.argsort(struct_importances * struct_values)[-5:][::-1]  # top 5 contributing
    print("Top structured features contributing to fraud:")
    for j in top_struct_idx:
        print(f"  {struct_feature_names[j]} = {struct_values[j]} (importance {struct_importances[j]:.6f})")
    
    # SVD contribution (rough)
    job_svd = X_scraped_tfidf_svd[idx]
    svd_scores = job_svd * svd_importances  # weighted contribution
    top_svd_idx = np.argsort(svd_scores)[-3:][::-1]  # top 3 SVD components
    print("Top SVD components contributing to fraud and their top words:")
    for comp_idx in top_svd_idx:
        component = svd.components_[comp_idx]
        top_words_idx = np.argsort(component)[-5:][::-1]  # top 5 words
        top_words = tfidf.get_feature_names_out()[top_words_idx]
        print(f"  SVD component {comp_idx} (score {svd_scores[comp_idx]:.6f}): {', '.join(top_words)}")


--- Fraudulent Job #1 (index 2610) ---
Top structured features contributing to fraud:
  loc_len = 62.0 (importance 0.001532)
  has_state_code = 1.0 (importance 0.005379)
  industry_group_Other Business/Services = 1.0 (importance 0.003498)
  location_legitimacy = 5.0 (importance 0.000324)
  contains_digits = 1.0 (importance 0.001037)
Top SVD components contributing to fraud and their top words:
  SVD component 299 (score 0.000000): jackson, land, editor, event, pricing
  SVD component 93 (score 0.000000): established 1981, 1981, established 1993, 1993, established 2008
  SVD component 95 (score 0.000000): established 1973, 1973, established 1986, 1986, established 2002

--- Fraudulent Job #2 (index 6909) ---
Top structured features contributing to fraud:
  loc_len = 19.0 (importance 0.001532)
  has_state_code = 1.0 (importance 0.005379)
  location_legitimacy = 1.0 (importance 0.000324)
  industry_group_Transportation = 0.0 (importance 0.000000)
  industry_group_Education = 0.0 (importa